# DroneAI Stage 1 data gate

This notebook imports the official 606 MB UP-COUNT sequence-sample pack into Google Drive, normalizes matching point labels, and scores rights, file integrity, annotation bounds, split leakage and condition coverage. It does not train a model. UP-COUNT is restricted to non-commercial research use.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/DroneAI')
PILOT_ROOT = DRIVE_ROOT / 'datasets' / 'up-count-sequence-samples-v1'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
print(PILOT_ROOT)

In [ ]:
import os, stat, subprocess
from google.colab import userdata
REPO_URL = 'https://github.com/LuciTa81/DroneAI.git'
REPO_DIR = Path('/content/DroneAI')
token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Add the GITHUB_TOKEN secret and grant notebook access.')
askpass = Path('/tmp/droneai_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'password' in prompt else 'x-access-token')\n", encoding='utf-8')
askpass.chmod(askpass.stat().st_mode | stat.S_IEXEC)
git_env = os.environ.copy()
git_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': token})
try:
    if (REPO_DIR / '.git').is_dir():
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], env=git_env, check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', 'main'], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], env=git_env, check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], env=git_env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    del token
    git_env.pop('GITHUB_TOKEN', None)
subprocess.run(['git', '-C', str(REPO_DIR), 'status', '--short', '--branch'], check=True)

In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR, check=True)

In [ ]:
import shutil
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print({'total_GB': round(total/1e9, 2), 'used_GB': round(used/1e9, 2), 'free_GB': round(free/1e9, 2)})
if free < 2_000_000_000:
    raise RuntimeError('Keep at least 2 GB free before importing the 606 MB pilot pack.')

In [ ]:
import gdown, urllib.request
sample_zip = PILOT_ROOT / 'examples_of_sequences.zip'
if not sample_zip.exists():
    result = gdown.download(id='1mvNxzHJzmHJsk9eYA6EwjpyY--XVas9U', output=str(sample_zip), quiet=False)
    if not result:
        raise RuntimeError('UP-COUNT sample download failed')
zenodo_base = 'https://zenodo.org/api/records/12683104/files'
labels_zip = PILOT_ROOT / 'labels.zip'
if not labels_zip.exists():
    urllib.request.urlretrieve(f'{zenodo_base}/labels.zip/content', labels_zip)
split_dir = PILOT_ROOT / 'splits'
split_dir.mkdir(exist_ok=True)
for name in ('train.txt', 'val.txt', 'test.txt'):
    target = split_dir / name
    if not target.exists():
        urllib.request.urlretrieve(f'{zenodo_base}/{name}/content', target)
print({'sample_zip_MB': round(sample_zip.stat().st_size/1e6, 2), 'labels_zip_MB': round(labels_zip.stat().st_size/1e6, 2)})

In [ ]:
import zipfile
sample_dir = PILOT_ROOT / 'raw_samples'
sample_dir.mkdir(exist_ok=True)
if not any(sample_dir.rglob('*.jpg')):
    with zipfile.ZipFile(sample_zip) as archive:
        archive.extractall(sample_dir)
temp_label_dir = Path('/content/upcount-labels')
if temp_label_dir.exists():
    shutil.rmtree(temp_label_dir)
with zipfile.ZipFile(labels_zip) as archive:
    archive.extractall(temp_label_dir)
images = [p for p in sample_dir.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
labels = list(temp_label_dir.rglob('*.txt'))
print({'sample_images': len(images), 'available_labels': len(labels)})

In [ ]:
subprocess.run([
    sys.executable, 'scripts/prepare_up_count.py',
    '--dataset-root', str(PILOT_ROOT),
    '--image-root', str(sample_dir),
    '--label-root', str(temp_label_dir),
    '--split-dir', str(split_dir),
    '--allow-image-subset',
], cwd=REPO_DIR, check=True)
result = subprocess.run([
    sys.executable, 'scripts/run_stage1.py',
    '--manifest', 'configs/datasets/up_count.sample.research.json',
    '--dataset-root', str(PILOT_ROOT),
    '--output-dir', str(DRIVE_ROOT / 'runs' / 'stage-1'),
], cwd=REPO_DIR, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print('Exit code:', result.returncode)
if result.returncode != 0:
    raise RuntimeError('Stage 1 gate did not pass; inspect the failed blockers above.')

In [ ]:
import json, pandas as pd
from IPython.display import Markdown, display
report_dir = DRIVE_ROOT / 'runs' / 'stage-1'
score = json.loads((report_dir / 'score.json').read_text(encoding='utf-8'))
display(Markdown((report_dir / 'score.md').read_text(encoding='utf-8')))
display(pd.DataFrame(score['checks'])[['category', 'description', 'weight', 'earned', 'blocker', 'observed']])
print('Decision:', score['status'], score['score'], '/ 100')